<img src=https://courseware.decoded.com/images/decoded/logo-decoded.png align=left width=100px/>

# M07 Rapid building machine learning workflow - Rapid EDA

---
<small>© 2026 Decoded Limited. All rights reserved. Website: https://decoded.com </small>

### Rapid EDA with ydata-profiling
Welcome! Exploratory Data Analysis (EDA) is a crucial first step in any machine learning project. However, writing individual lines of code to check distributions, correlations, and outliers for *every single column* can be incredibly tedious.

Enter **`ydata-profiling`**. In just a few lines of code, it generates a comprehensive, interactive HTML report of your dataset. 

In this notebook, we will use it to perform rapid EDA on the **Juice** (Classification) and **Insurance** (Regression) datasets we used in our previous PyCaret modeling modules.

In [1]:
# Install dependencies (uncomment if running for the first time in this environment)
# !pip install ydata-profiling pycaret

In [2]:
# Import libraries 
from pycaret.datasets import get_data
from ydata_profiling import ProfileReport

## 0. Loading The Data From A SQLlite Database

#### Connecting the database

In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data_class.db")
cursor = conn.cursor()

#### What does this database contain?

In [6]:
# 2. Get a list of all tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

for table in tables:
    print(table[0])

purchases


Now we know the tables found in this database. Let's investigate more about the table

## 1. Loading the Data


In [7]:
### Import all rows from the table using read_sql in pandas
juice_data = pd.read_sql("SELECT * FROM purchases", conn)

In [8]:
#Load Classification Data
print("\nLoading Juice Dataset...")
juice_data.head()


Loading Juice Dataset...


,Id,Purchase,WeekofPurchase,StoreID,PriceCH,PriceMM,DiscCH,DiscMM,SpecialCH,SpecialMM,LoyalCH,SalePriceMM,SalePriceCH,PriceDiff,Store7,PctDiscMM,PctDiscCH,ListPriceDiff,STORE
0,1,CH,237,1,1.75,1.99,0.00,0.0,0,0,0.500000,1.99,1.75,0.24,No,0.000000,0.000000,0.24,1
1,2,CH,239,1,1.75,1.99,0.00,0.3,0,1,0.600000,1.69,1.75,-0.06,No,0.150754,0.000000,0.24,1
2,3,CH,245,1,1.86,2.09,0.17,0.0,0,0,0.680000,2.09,1.69,0.40,No,0.000000,0.091398,0.23,1
3,4,MM,227,1,1.69,1.69,0.00,0.0,0,0,0.400000,1.69,1.69,0.00,No,0.000000,0.000000,0.00,1
4,5,CH,228,7,1.69,1.69,0.00,0.0,0,0,0.956535,1.69,1.69,0.00,Yes,0.000000,0.000000,0.00,0


### Simple ETL commands in SQL

In [9]:
# maybe show some basic SQL commands he
# Transform: Count purchases made per store

transform_query = """
SELECT StoreID, COUNT(Purchase) AS total_rentals
FROM purchases
GROUP BY StoreID
"""
results = cursor.execute(transform_query).fetchall()
results[0:5]


[(1, 157), (2, 222), (3, 196), (4, 139), (7, 356)]

In [10]:
# Load: Save into a new summary table
cursor.execute("""
CREATE TABLE IF NOT EXISTS StoreDetail (
    StoreID INTEGER,
    StoreName INTEGER
)
""")

In [11]:
# Check if the table has been created

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

for table in tables:
    print(table[0])

purchases
StoreDetail


## 2. Generating the Profile Reports
Instead of writing dozens of `df.describe()` and `df.hist()` commands, we pass our dataframes into `ProfileReport`.

*Note: To save memory in the notebook, we will export the report as an HTML file.*

In [7]:
# --- Profile the Juice Dataset ---
juice_profile = ProfileReport(juice_data, title="Juice Dataset EDA Report", explorative=True)

# Display interactively inside the notebook
juice_profile.to_file("juice_eda_report.html")
print("\nJuice EDA Report saved as 'juice_eda_report.html'")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 19/19 [00:00<00:00, 283600.63it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]


Juice EDA Report saved as 'juice_eda_report.html'




#### Activity 1: The Alert Detective (Juice Dataset)

**Objective:** Learn to navigate the `ydata-profiling` report and identify multicollinearity.

**Instructions:**

1. Run the `ydata-profiling` cell for the **Juice** dataset from your notebook.
2. Open the **Alerts** tab (usually found at the very top of the report).
3. Scroll down to the **Correlations** section and look at the heatmap (toggle between Spearman and Pearson).

**Tasks:**

* Which two variables are the *most* highly correlated with `PriceCH` (Price of Citrus Hill)?
* Look at `ListPriceDiff`. Based on its relationship with other price columns, why might the profiler flag it as highly correlated or redundant?
* In your own words, why is it dangerous to leave perfectly correlated features (like a base price and a calculated discount) in a dataset before feeding it to a machine learning model?

---


### Activity 2: Diagnosing Distribution Skew (Insurance Dataset)

**Objective:** Understand how target variable distributions impact modeling decisions.

**Instructions:**

1. Open the generated `insurance_eda_report.html` file in your web browser.
2. Navigate to the **Variables** section and find the `charges` column (this is our target variable for regression).
3. Click **Toggle details** for the `charges` column and look at the Histogram and Extreme Values.

**Questions to Answer:**

* How would you describe the shape of the `charges` distribution? (e.g., Normal, Left-Skewed, Right-Skewed, Bimodal?)
* Look at the "Extreme Values" tab for `charges`. What is the difference between the 95th percentile charge and the maximum charge?
* If we train a PyCaret regression model on this data *without* applying a transformation (like a log transform), how might these extreme high-charge patients skew our model's predictions for average patients?

## 3. Turning Insights into Action: Handling Collinearity & Skew
While PyCaret's datasets are mostly free of missing values, `ydata-profiling` will immediately alert you to other structural issues in the **Alerts** tab. 

### Expected Alerts & How to Fix Them
* **High Correlation (Multicollinearity) in Juice:** You will likely see alerts that variables like `PriceCH` (Price of Citrus Hill) and `SalePriceCH` are highly correlated. 
    * *Action:* **Drop** highly correlated or derived features to prevent multicollinearity, which can confuse linear models.
* **Skewed Distributions in Insurance:** The `charges` variable is highly right-skewed (most people have low charges, a few have very high charges).
    * *Action:* While we cannot drop our target variable, recognizing this skew tells us we might need to apply a **log transformation** during our PyCaret setup to help our regression models perform better.

### Activity 3: Code the Cleanup

**Objective:** Translate visual EDA insights into functional Python code.

**Instructions:**
You noticed in the Juice dataset report that `PctDiscCH` (Percent Discount Citrus Hill) and `PctDiscMM` (Percent Discount Minute Maid) are perfectly derived from the price columns, causing heavy multicollinearity.

**Your Task:**
Write a quick Python script using `pandas` to:

1. Drop the `PctDiscCH` and `PctDiscMM` columns.
2. Print the final shape of the dataframe to verify they were removed.

> **Stretch** Can you also write a line of code to check if there are any actual missing values (`NaN`) left in your newly cleaned dataframe?

In [ ]:
# --- Executing a Targeted Data Cleaning Strategy ---

# 1. Clean the Juice Data (Dropping highly correlated/redundant columns)
juice_clean = juice_data.copy()
# We drop 'PctDiscCH' and 'PctDiscMM' because they are perfectly derived from the price and discount columns
columns_to_drop = ['PctDiscCH', 'PctDiscMM']
juice_clean = juice_clean.drop(columns=columns_to_drop)

print(f"Original Juice Shape: {juice_data.shape}")
print(f"Cleaned Juice Shape: {juice_clean.shape}\n")

# 2. Prep the Insurance Data (Making a note for PyCaret Setup)
# Instead of manually transforming the 'charges' column here, 
# we now know to pass `transform_target=True` into our PyCaret Regression setup later!
print("Insight recorded: Use transform_target=True in PyCaret Regression setup to handle 'charges' skewness.")

## Summary
You just went from raw data to actionable statistical insights in seconds. 

By checking the **Alerts** and **Correlations** tabs in the profile report, you identified multicollinearity to remove manually.